# Smile Metric Evaluation

## 1. Import Libraries and Setup

In [1]:
import os
import py_vncorenlp

def set_up_visegmenter():
    """Initialize VnCoreNLP word segmenter for Vietnamese text."""
    vncorenlp_dir = '/home/vlai-vqa-nle/minhtq/vqa-nle/vncorenlp_models'
    if not os.path.exists(vncorenlp_dir):
        os.makedirs(vncorenlp_dir)

    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg"], 
        save_dir='/home/vlai-vqa-nle/minhtq/vqa-nle/src/inference/vncorenlp_models'
    )
    return rdrsegmenter

def segment_text(text: str, rdrsegmenter) -> str:
    """Segment Vietnamese text using VnCoreNLP."""
    if not text:
        return ""
    try:
        sentences = rdrsegmenter.word_segment(text)
        segmented_text = " ".join([" ".join(sentence) for sentence in sentences])
        return segmented_text
    except Exception as e:
        print(f"Error segmenting text: {e}. Text: '{text}'. Returning original text.")
        return text

print("Setting up Vietnamese text segmenter...")
rdrsegmenter = set_up_visegmenter()
print("Vietnamese text segmenter initialized")

Setting up Vietnamese text segmenter...
2025-12-06 18:51:58 INFO  WordSegmenter:24 - Loading Word Segmentation model
Vietnamese text segmenter initialized


In [2]:
import json
import re
import numpy as np
import sys
from pathlib import Path

smile_path = '/home/vlai-vqa-nle/minhtq/vqa-nle/smile-metric-qna-eval'
if smile_path not in sys.path:
    sys.path.append(smile_path)

from smile.smile import SMILE

print("Libraries imported successfully")

/opt/miniconda3/envs/project_vivqanle_grpo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/vlai-vqa-nle/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /home/vlai-vqa-
[nltk_data]     nle/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/vlai-vqa-
[nltk_data]     nle/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## 2. Load Data from completions.jsonl

In [3]:
completions_path = '/home/vlai-vqa-nle/minhtq/vqa-nle/ms-swift/examples/train/grpo/output/only_think_answer/v0-20251126-164025/completions.jsonl'

data_list = []
with open(completions_path, 'r', encoding='utf-8') as f:
    for line in f:
        data_list.append(json.loads(line))

print(f"Loaded {len(data_list)} samples from completions.jsonl")
print(f"Key: {list(data_list[0].keys())}")

Loaded 1001 samples from completions.jsonl
Key: ['step', 'prompt', 'completion', 'CustomFormatReward_ViVQA_X_Only_Think_Answer', 'CustomAccuracyReward', 'CustomExplainationRewardOnlyThinkAnswer', 'advantages', 'solution']


## 3. Extract Questions, Ground Truth Answers, and Predictions

In [4]:
def extract_answer_from_tags(text):
    """
    Extract answer from tags like <answer></answer>
    """
    if not text:
        return ""
    if isinstance(text, list):
        text = text[0] if text else ""
    
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

def extract_question_from_prompt(prompt):
    """
    Extract question from the prompt field
    """
    if not prompt:
        return ""
    if isinstance(prompt, list):
        prompt = prompt[0] if prompt else ""
    
    match = re.search(r'Câu hỏi:\s*(.+?)\s*\n\s*Vui lòng', prompt, re.DOTALL)
    if match:
        return match.group(1).strip()

    match = re.search(r'Câu hỏi:\s*(.+?)\s*Câu trả lời:', prompt, re.DOTALL)
    if match:
        return match.group(1).strip()
    
    match = re.search(r'Câu hỏi:\s*(.+?)[\?\n]', prompt, re.DOTALL)
    if match:
        return match.group(1).strip() + "?"
    
    return "Question not found"

extracted_data = []
for item in data_list:
    question = extract_question_from_prompt(item.get('prompt', ''))
    ground_truth = extract_answer_from_tags(item.get('solution', ''))
    prediction = extract_answer_from_tags(item.get('completion', ''))
    
    if question and ground_truth and prediction:
        extracted_data.append({
            'question': question,
            'ground_truth': ground_truth,
            'prediction': prediction
        })

print(f"Successfully extracted {len(extracted_data)} samples with complete data")
print(f"\nSample Data(First Item)")
print(f"Question: {extracted_data[0]['question']}")
print(f"Ground Truth: {extracted_data[0]['ground_truth']}")
print(f"Prediction: {extracted_data[0]['prediction']}")

Successfully extracted 978 samples with complete data

Sample Data(First Item)
Question: Bếp sử dụng gas hay điện?
Ground Truth: gas
Prediction: Gas


## 4. Create synthetic answers

In [5]:
DEFAULT_SYSTEM_PROMPT = """You are an intelligent assistant that converts short answers into complete sentences. 
Given a question and a short answer, generate a single sentence that incorporates the answer. 
Use the exact words from the provided answer. Do not include the question in your response. 
Return only the sentence, nothing else."""

DEFAULT_USER_PROMPT_TEMPLATE = """Question: {question}
Answer: {answer}

Generate a single sentence answer:"""

# Vietnamese 
VI_SYSTEM_PROMPT = """Bạn là một trợ lý thông minh có nhiệm vụ chuyển đổi các câu trả lời ngắn thành các câu hoàn chỉnh. 
Khi nhận được một câu hỏi và một câu trả lời ngắn, hãy tạo ra một câu đơn duy nhất có chứa nội dung câu trả lời đó. 
Hãy sử dụng chính xác các từ ngữ trong câu trả lời đã cho. Không được đưa câu hỏi vào trong phản hồi của bạn. 
Chỉ trả về duy nhất câu kết quả, không thêm bất kỳ nội dung nào khác."""

VI_USER_PROMPT_TEMPLATE = """Câu hỏi: {question}
Câu trả lời: {answer}

Hãy tạo ra một câu trả lời dạng câu đơn:"""

### Vintern

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [8]:
notebooks_dir = '/home/vlai-vqa-nle/minhtq/vqa-nle/notebooks'
if notebooks_dir not in sys.path:
    sys.path.insert(0, notebooks_dir)

from synthetic_answer_generator import *

In [9]:
model_path = "/home/vlai-vqa-nle/.cache/huggingface/hub/models--5CD-AI--Vintern-3B-R-beta/snapshots/4fd34d713dfca446cdecc00d921f5038909e3efb"
model, tokenizer, device = load_vintern_model(model_path = model_path, device = device)
# model, tokenizer, device = load_qwen2_vl_model(device = device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Vintern model from: /home/vlai-vqa-nle/.cache/huggingface/hub/models--5CD-AI--Vintern-3B-R-beta/snapshots/4fd34d713dfca446cdecc00d921f5038909e3efb
Using device: cuda


/opt/miniconda3/envs/project_vivqanle_grpo/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention2 is not installed.


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  8.32it/s]


Vintern model loaded successfully!


In [ ]:
data_list = extracted_data[:10]
# user_prompt_template = VI_USER_PROMPT_TEMPLATE
# system_prompt = VI_SYSTEM_PROMPT
user_prompt_template = DEFAULT_USER_PROMPT_TEMPLATE
system_prompt = DEFAULT_SYSTEM_PROMPT
synthetic_answer = generate_synthetic_answers_batch_vintern(model = model, tokenizer = tokenizer, device = device, data_list = data_list, max_new_tokens = 256, user_prompt_template = user_prompt_template, system_prompt = system_prompt)

Generating synthetic answers: 100%|██████████| 10/10 [00:04<00:00,  2.21it/s]


In [ ]:
#vi prompt
for answer in synthetic_answer:
    print(answer)

{'question': 'Bếp sử dụng gas hay điện?', 'ground_truth': 'gas', 'prediction': 'Gas', 'syn_ans': 'Bếp sử dụng gas.'}
{'question': 'Môn thể thao nào được trình chiếu?', 'ground_truth': 'trượt tuyết', 'prediction': 'Trượt tuyết', 'syn_ans': 'Trượt tuyết được trình chiếu.'}
{'question': 'Có phải anh ấy đang thực hiện một cú nhảy nguy hiểm?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng.'}
{'question': 'Người phụ nữ có cầm máy ảnh không?', 'ground_truth': 'có', 'prediction': 'Có', 'syn_ans': 'Có.'}
{'question': 'Đây có phải là một chương trình giải thưởng?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng.'}
{'question': 'Sàn nhà trong phòng tắm này là gì?', 'ground_truth': 'gạch', 'prediction': 'Gạch vuông', 'syn_ans': 'Sàn nhà trong phòng tắm này là gạch.'}
{'question': 'Cậu bé đang làm gì?', 'ground_truth': 'trượt ván', 'prediction': 'Trượt băng', 'syn_ans': 'Cậu bé đang trượt ván.'}
{'question': 'Nước dường như đang chuyển động?', 'ground_truth': 'đúng', 'p

In [ ]:
#english prompt
for answer in synthetic_answer:
    print(answer)

{'question': 'Bếp sử dụng gas hay điện?', 'ground_truth': 'gas', 'prediction': 'Gas', 'syn_ans': 'Bếp sử dụng gas.'}
{'question': 'Môn thể thao nào được trình chiếu?', 'ground_truth': 'trượt tuyết', 'prediction': 'Trượt tuyết', 'syn_ans': 'Môn thể thao được trình chiếu là trượt tuyết.'}
{'question': 'Có phải anh ấy đang thực hiện một cú nhảy nguy hiểm?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng, anh ấy đang thực hiện một cú nhảy nguy hiểm.'}
{'question': 'Người phụ nữ có cầm máy ảnh không?', 'ground_truth': 'có', 'prediction': 'Có', 'syn_ans': 'Có.'}
{'question': 'Đây có phải là một chương trình giải thưởng?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đây là một chương trình giải thưởng.'}
{'question': 'Sàn nhà trong phòng tắm này là gì?', 'ground_truth': 'gạch', 'prediction': 'Gạch vuông', 'syn_ans': 'Sàn nhà trong phòng tắm này được làm bằng gạch.'}
{'question': 'Cậu bé đang làm gì?', 'ground_truth': 'trượt ván', 'prediction': 'Trượt băng', 'syn_ans'

### Qwen3-4B-Instruct

In [12]:
model_path = "/mnt/dataset1/pretrained_fm/Qwen_Qwen3-4B-Instruct-2507"
model, processor, device = load_qwen_text_model(model_path=model_path, device=device)

Loading Qwen text model from: /mnt/dataset1/pretrained_fm/Qwen_Qwen3-4B-Instruct-2507
Using device: cuda


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.06it/s]


Qwen text model loaded successfully!


In [15]:
data_list = extracted_data[:10]
user_prompt_template = DEFAULT_USER_PROMPT_TEMPLATE
system_prompt = DEFAULT_SYSTEM_PROMPT
# user_prompt_template = VI_USER_PROMPT_TEMPLATE
# system_prompt = VI_SYSTEM_PROMPT
synthetic_answer = generate_synthetic_answers_batch_qwen(model = model, processor = processor, device = device, data_list = data_list, max_new_tokens = 256)

Generating synthetic answers: 100%|██████████| 10/10 [00:04<00:00,  2.23it/s]


In [ ]:
#vi prompt
for answer in synthetic_answer:
    print(answer)

{'question': 'Bếp sử dụng gas hay điện?', 'ground_truth': 'gas', 'prediction': 'Gas', 'syn_ans': 'Bếp sử dụng gas.'}
{'question': 'Môn thể thao nào được trình chiếu?', 'ground_truth': 'trượt tuyết', 'prediction': 'Trượt tuyết', 'syn_ans': 'Môn thể thao được trình chiếu là trượt tuyết.'}
{'question': 'Có phải anh ấy đang thực hiện một cú nhảy nguy hiểm?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng.'}
{'question': 'Người phụ nữ có cầm máy ảnh không?', 'ground_truth': 'có', 'prediction': 'Có', 'syn_ans': 'Có, người phụ nữ có cầm máy ảnh.'}
{'question': 'Đây có phải là một chương trình giải thưởng?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng.'}
{'question': 'Sàn nhà trong phòng tắm này là gì?', 'ground_truth': 'gạch', 'prediction': 'Gạch vuông', 'syn_ans': 'Sàn nhà trong phòng tắm này là gạch.'}
{'question': 'Cậu bé đang làm gì?', 'ground_truth': 'trượt ván', 'prediction': 'Trượt băng', 'syn_ans': 'Cậu bé đang trượt ván.'}
{'question': 'Nước dường như đ

In [16]:
#english prompt
for answer in synthetic_answer:
    print(answer)

{'question': 'Bếp sử dụng gas hay điện?', 'ground_truth': 'gas', 'prediction': 'Gas', 'syn_ans': 'Bếp sử dụng gas.'}
{'question': 'Môn thể thao nào được trình chiếu?', 'ground_truth': 'trượt tuyết', 'prediction': 'Trượt tuyết', 'syn_ans': 'Môn thể thao được trình chiếu là trượt tuyết.'}
{'question': 'Có phải anh ấy đang thực hiện một cú nhảy nguy hiểm?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng.'}
{'question': 'Người phụ nữ có cầm máy ảnh không?', 'ground_truth': 'có', 'prediction': 'Có', 'syn_ans': 'Có, người phụ nữ có cầm máy ảnh.'}
{'question': 'Đây có phải là một chương trình giải thưởng?', 'ground_truth': 'đúng', 'prediction': 'Có', 'syn_ans': 'Đúng.'}
{'question': 'Sàn nhà trong phòng tắm này là gì?', 'ground_truth': 'gạch', 'prediction': 'Gạch vuông', 'syn_ans': 'Sàn nhà trong phòng tắm này là gạch.'}
{'question': 'Cậu bé đang làm gì?', 'ground_truth': 'trượt ván', 'prediction': 'Trượt băng', 'syn_ans': 'Cậu bé đang trượt ván.'}
{'question': 'Nước dường như đ

## 5. Format Data for Smile Metric

The Smile metric expects data in the format: `(question, answer, syn_ans, pred)`

Since we want to use only "answer" instead of "syn_ans", we'll use the ground truth answer for both the `answer` and `syn_ans` fields.

In [ ]:
# note: Segment Vietnamese text before passing to PhoBERT
smile_data = []
for item in synthetic_answer:
    question_seg = segment_text(item['question'], rdrsegmenter)
    ground_truth_seg = segment_text(item['ground_truth'], rdrsegmenter)
    prediction_seg = segment_text(item['prediction'], rdrsegmenter)
    syn_ans_seg = segment_text(item['syn_ans'], rdrsegmenter)
    
    smile_data.append((
        question_seg,           # segmented question
        ground_truth_seg,       # segmented answer
        syn_ans_seg,       # segmented syn_ans, using same as answer
        prediction_seg          # segmented pred
    ))


smile_data_array = np.array(smile_data)
print(f"Formatted {smile_data_array.shape[0]} samples for Smile metric")
print(f"Data shape: {smile_data_array.shape}")
print(f"\n---Formatted Sample (First Item) ---")
print(f"Question: {smile_data_array[0, 0]}")
print(f"Answer: {smile_data_array[0, 1]}")
print(f"Syn_ans: {smile_data_array[0, 2]}")
print(f"Prediction: {smile_data_array[0, 3]}")

Formatted 10 samples for Smile metric
Data shape: (10, 4)

---Formatted Sample (First Item) ---
Question: B ế p   s ử _ d ụ n g   g a s   h a y   đ i ệ n   ?
Answer: g a s
Syn_ans: B ế p   s ử _ d ụ n g   g a s   .
Prediction: G a s


## 6. Initialize Smile Metric

In [ ]:
smile = SMILE(
    emb_model='phobert',           # Embedding model
    eval_metrics=['avg', 'hm'],     # Average and Harmonic Mean metrics
    assign_bins=False,              # Don't assign bins
    use_exact_matching=True,        # Use exact matching for keyword scores
    verbose=True                    # Show progress
)

print("SMILE metric initialized successfully")

Loading PhoBERT from local path: /mnt/dataset1/pretrained_fm/vinai/phobert-base


PhoBERT loaded successfully on cuda
SMILE metric initialized successfully


## 7. Compute Smile Scores

In [ ]:
print("Computing Smile scores...\n")
results = smile.generate_scores(smile_data_array)

print("\nSmile scores computed successfully!")

Computing Smile scores...



 > Generating sentence embeddings...


Encoding: 100%|██████████| 1/1 [00:00<00:00, 87.12it/s]

 > Generating sent_emb_scores...
 > Generating kwd_emb_scores...



Smile scores computed successfully!


## 8. Display Results

In [ ]:
# Sentence Embedding Scores
sent_scores = results['sent_emb_scores']
print(f"\n📊 Sentence Embedding Scores:")
print(f"  Mean:  {np.mean(sent_scores):.4f}")
print(f"  Std:   {np.std(sent_scores):.4f}")
print(f"  Min:   {np.min(sent_scores):.4f}")
print(f"  Max:   {np.max(sent_scores):.4f}")

# Keyword Scores
kwd_scores = results['kwd_scores']
print(f"\n🔑 Keyword Scores:")
print(f"  Mean:  {np.mean(kwd_scores):.4f}")
print(f"  Std:   {np.std(kwd_scores):.4f}")
print(f"  Min:   {np.min(kwd_scores):.4f}")
print(f"  Max:   {np.max(kwd_scores):.4f}")

# Average Scores
avg_scores = results['avg']
print(f"\n😊 SMILE Average Scores:")
print(f"  Mean:  {np.mean(avg_scores):.4f}")
print(f"  Std:   {np.std(avg_scores):.4f}")
print(f"  Min:   {np.min(avg_scores):.4f}")
print(f"  Max:   {np.max(avg_scores):.4f}")

# Harmonic Mean Scores
hm_scores = results['hm']
print(f"\n🤝 SMILE Harmonic Mean Scores:")
print(f"  Mean:  {np.mean(hm_scores):.4f}")
print(f"  Std:   {np.std(hm_scores):.4f}")
print(f"  Min:   {np.min(hm_scores):.4f}")
print(f"  Max:   {np.max(hm_scores):.4f}")


📊 Sentence Embedding Scores:
  Mean:  0.6814
  Std:   0.1760
  Min:   0.3927
  Max:   0.9162

🔑 Keyword Scores:
  Mean:  0.6742
  Std:   0.3317
  Min:   0.2654
  Max:   1.0000

😊 SMILE Average Scores:
  Mean:  0.6778
  Std:   0.2289
  Min:   0.3291
  Max:   0.9371

🤝 SMILE Harmonic Mean Scores:
  Mean:  0.6468
  Std:   0.2426
  Min:   0.3167
  Max:   0.9329


## 8. Display Individual Sample Scores (First 5 Samples)

In [ ]:
num_samples_to_show = min(5, len(extracted_data))
for i in range(num_samples_to_show):
    print(f"\n--- Sample {i+1} ---")
    print(f"Question: {extracted_data[i]['question'][:80]}...")
    print(f"Ground Truth: {extracted_data[i]['ground_truth']}")
    print(f"Prediction: {extracted_data[i]['prediction']}")
    print(f"Sent Emb Score: {sent_scores[i]:.4f}")
    print(f"Keyword Score: {kwd_scores[i]:.4f}")
    print(f"SMILE Avg: {avg_scores[i]:.4f}")
    print(f"SMILE HM: {hm_scores[i]:.4f}")
    if 'max_sim_words' in results:
        print(f"Max Sim Word: {results['max_sim_words'][i]}")


--- Sample 1 ---
Question: Bếp sử dụng gas hay điện?...
Ground Truth: gas
Prediction: Gas
Sent Emb Score: 0.6427
Keyword Score: 1.0000
SMILE Avg: 0.8214
SMILE HM: 0.7825
Max Sim Word: g a s

--- Sample 2 ---
Question: Môn thể thao nào được trình chiếu?...
Ground Truth: trượt tuyết
Prediction: Trượt tuyết
Sent Emb Score: 0.9162
Keyword Score: 0.8636
SMILE Avg: 0.8899
SMILE HM: 0.8891
Max Sim Word: t r ư ợ t _ t u y ế t

--- Sample 3 ---
Question: Có phải anh ấy đang thực hiện một cú nhảy nguy hiểm?...
Ground Truth: đúng
Prediction: Có
Sent Emb Score: 0.5845
Keyword Score: 0.2812
SMILE Avg: 0.4329
SMILE HM: 0.3797
Max Sim Word: c ó

--- Sample 4 ---
Question: Người phụ nữ có cầm máy ảnh không?...
Ground Truth: có
Prediction: Có
Sent Emb Score: 0.4970
Keyword Score: 1.0000
SMILE Avg: 0.7485
SMILE HM: 0.6640
Max Sim Word: c ó

--- Sample 5 ---
Question: Đây có phải là một chương trình giải thưởng?...
Ground Truth: đúng
Prediction: Có
Sent Emb Score: 0.5845
Keyword Score: 0.2812
SMILE Avg:

## 9. Save Results 

In [ ]:
# import pickle

# output_path = '/home/vlai-vqa-nle/phatdat/notebooks/smile_results.pkl'

# with open(output_path, 'wb') as f:
#     pickle.dump(results, f)

# print(f"Results saved to: {output_path}")

# summary = {
#     'num_samples': len(extracted_data),
#     'sent_emb_scores': {
#         'mean': float(np.mean(sent_scores)),
#         'std': float(np.std(sent_scores)),
#         'min': float(np.min(sent_scores)),
#         'max': float(np.max(sent_scores))
#     },
#     'keyword_scores': {
#         'mean': float(np.mean(kwd_scores)),
#         'std': float(np.std(kwd_scores)),
#         'min': float(np.min(kwd_scores)),
#         'max': float(np.max(kwd_scores))
#     },
#     'smile_avg': {
#         'mean': float(np.mean(avg_scores)),
#         'std': float(np.std(avg_scores)),
#         'min': float(np.min(avg_scores)),
#         'max': float(np.max(avg_scores))
#     },
#     'smile_hm': {
#         'mean': float(np.mean(hm_scores)),
#         'std': float(np.std(hm_scores)),
#         'min': float(np.min(hm_scores)),
#         'max': float(np.max(hm_scores))
#     }
# }

# summary_path = '/home/vlai-vqa-nle/phatdat/notebooks/smile_summary.json'
# with open(summary_path, 'w', encoding='utf-8') as f:
#     json.dump(summary, f, indent=2, ensure_ascii=False)

# print(f"Summary saved to: {summary_path}")